## Importing Required Python Modules

In [111]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
import pickle
import numpy as np

## Loading the Dataset

In [112]:
## Load Dataset
df = pd.read_csv('./Churn_Modelling.csv')
df.head(10)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,10,15592389,H?,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


## Data Quality Check

In [113]:
## Checking Dataset Info
print("Shape of the Dataset:", df.shape, "\n")
df.info()

Shape of the Dataset: (10000, 14) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [114]:
## Checking for missing values
df.isna().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

## Data Preprocessing

In [115]:
## Dropping Unnecessary Columns - 'RowNumber', 'CustomerId', 'Surname'
print("Shape of the Dataset before dropping unnecessary columns for modeling:", df.shape, "\n\n")

## Drop the columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

##Checcking the shape after dropping the columns
print("Shape of the Dataset After dropping unnecessary columns for modeling:", df.shape, "\n\n")

## Display the first 10 rows of the modified dataframe
display(df.head(10))

Shape of the Dataset before dropping unnecessary columns for modeling: (10000, 14) 


Shape of the Dataset After dropping unnecessary columns for modeling: (10000, 11) 




,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


In [116]:
## Checking the DataTypes of the columns
df.dtypes

CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

- **We have 2 columns `Geography` & `Gender` having Datatype as Object.**
- **Need to Apply LabelEncoder on these columns before training the Model.**

In [117]:
## Checking for the Unique values in Categorical Columns
print(f"Total unique count & values in 'Geography' column: {df['Geography'].nunique()} & {df['Geography'].unique()}\n")
print(f"Total unique count & values in 'Gender' column: {df['Gender'].nunique()} & {df['Gender'].unique()}\n")

Total unique count & values in 'Geography' column: 3 & ['France' 'Spain' 'Germany']

Total unique count & values in 'Gender' column: 2 & ['Female' 'Male']



**LabelEncoder is usually a bad idea for nominal features with more than 2 categories in an ANN because it injects fake numeric structure into the data, while One‑Hot Encoding keeps categories independent and matches how dense layers learn.**


## Why LabelEncoder is problematic (>2 classes)

### When you use LabelEncoder on a feature like "France", "Spain", "Germany", you get something like:
France → 0
Spain → 1
Germany → 2

*For the network this means:*

It “sees” a single scalar feature whose value can be 0,1,2.
It can interpret distance and ordering: 2 is “larger” than 0, 1 is “between” 0 and 2, etc., even though these categories have no such order.
In the first dense layer, this scalar is just multiplied by a single weight www, so:
encoded = 0 → contribution 0⋅w0\cdot w0⋅w
encoded = 2 → contribution 2⋅w2\cdot w2⋅w
The model is forced to treat categories as lying on a one‑dimensional line, which is an arbitrary and usually wrong geometry for nominal data.

* This can lead to:
    - Unwanted bias: some categories are always “closer” or “further” than others for no semantic reason.
    - Harder optimization: the network must learn weird nonlinear transformations just to “undo” the artificial order before extracting useful patterns.
    
So LabelEncoder is only appropriate when the categories really are ordinal (e.g., “low < medium < high”), which is rarely the case for typical ANN inputs like country, city, product type, or color.

In [118]:
## Creating Label Encoders for Categorical Columns: "Gender"
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
display(df.head(10))

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
5,645,Spain,1,44,8,113755.78,2,1,0,149756.71,1
6,822,France,1,50,7,0.00,2,1,1,10062.80,0
7,376,Germany,0,29,4,115046.74,4,1,0,119346.88,1
8,501,France,1,44,4,142051.07,2,0,1,74940.50,0
9,684,France,1,27,2,134603.88,1,1,1,71725.73,0


In [119]:
## Creating One Hot Encoders for Categorical Columns: "Geography"
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
encoded_geography = ohe.fit_transform(df[['Geography']])

In [120]:
## Checking Features after OneHotEncoding
print('OneHotEncoded Data Features: \n')
display(ohe.get_feature_names_out(['Geography']))

OneHotEncoded Data Features: 



array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [121]:
print('OneHotEncoded Data will look alike: \n')
display(encoded_geography)


OneHotEncoded Data will look alike: 



array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [122]:
## Converting OneHotEncoded Data to a DataFrame to merge with original DataFrame
geography_df = pd.DataFrame(encoded_geography, columns=ohe.get_feature_names_out(['Geography']))
display(geography_df)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [123]:
## Concatinating the OneHotEncoded DataFrame with the original DataFrame
df = pd.concat([df.drop('Geography', axis=1),geography_df], axis=1)
print("Current shape of the dataframe:", df.shape) 
display(df.head(10))

Current shape of the dataframe: (10000, 13)


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
5,645,1,44,8,113755.78,2,1,0,149756.71,1,0.0,0.0,1.0
6,822,1,50,7,0.00,2,1,1,10062.80,0,1.0,0.0,0.0
7,376,0,29,4,115046.74,4,1,0,119346.88,1,0.0,1.0,0.0
8,501,1,44,4,142051.07,2,0,1,74940.50,0,1.0,0.0,0.0
9,684,1,27,2,134603.88,1,1,1,71725.73,0,1.0,0.0,0.0


In [124]:
## Splitting the dataset into Features and Target variable
X = df.drop('Exited',axis=1)
y = df['Exited']

## Splitting the dataset into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2,random_state=42)
print("Shape of splitted Data:", [X_train.shape, X_test.shape, y_train.shape, y_test.shape])

Shape of splitted Data: [(8000, 12), (2000, 12), (8000,), (2000,)]


In [125]:
## Scalling the Train & Test set
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Scaling completed.")

Scaling completed.


In [126]:
print("Checking the scaled data samples: \n")
display(X_train[:10])
display(X_test[:10])

Checking the scaled data samples: 



array([[ 0.24626866,  0.        , -0.41666667,  0.25      , -0.75579118,
         1.        ,  0.        ,  0.        ,  0.80020529,  1.        ,
         0.        ,  0.        ],
       [-0.15671642,  0.        ,  0.41666667, -0.25      ,  0.18162243,
         1.        ,  0.        ,  0.        ,  0.97210089,  0.        ,
         1.        ,  0.        ],
       [-0.70149254,  0.        , -1.08333333, -0.5       ,  0.14334464,
         0.        ,  0.        , -1.        , -0.14859457,  0.        ,
         0.        ,  1.        ],
       [-0.68656716, -1.        , -0.83333333,  1.        ,  0.30710031,
         0.        ,  0.        , -1.        ,  0.53539408,  1.        ,
         0.        ,  0.        ],
       [-1.01492537,  0.        ,  1.58333333,  1.        ,  0.3581171 ,
         0.        , -1.        , -1.        , -0.62097744,  1.        ,
         0.        ,  0.        ],
       [-0.62686567,  0.        ,  0.33333333, -0.25      ,  0.18647223,
         0.        ,  

array([[-0.42537313,  0.        , -0.41666667, -0.5       ,  0.00204958,
         1.        , -1.        , -1.        , -0.59756005,  0.        ,
         1.        ,  0.        ],
       [-0.2238806 ,  0.        ,  0.5       , -1.        , -0.75579118,
         1.        ,  0.        ,  0.        ,  0.46717681,  1.        ,
         0.        ,  0.        ],
       [-0.3880597 , -1.        ,  0.58333333, -0.25      , -0.75579118,
         1.        ,  0.        , -1.        , -0.42681133,  0.        ,
         0.        ,  1.        ],
       [-1.09701493,  0.        ,  1.83333333,  0.75      ,  0.17791978,
         1.        ,  0.        ,  0.        ,  0.71455556,  0.        ,
         1.        ,  0.        ],
       [-0.69402985, -1.        , -0.83333333,  0.5       ,  0.22371414,
         0.        ,  0.        ,  0.        ,  0.14437363,  0.        ,
         0.        ,  1.        ],
       [ 1.02238806,  0.        ,  0.        ,  0.75      , -0.75579118,
         1.        ,  

In [127]:
## Saving the LabelEncoder, OneHotEncoder and Scaler using pickle
# Saving LabelEncoder
with open('encoder_gender.pkl','wb') as f:
    pickle.dump(le,f)

# Saving OneHotEncoder
with open('encoder_geography.pkl','wb') as f:
    pickle.dump(ohe,f)

# Saving StandardScaler
with open('scaler.pkl','wb') as f:
    pickle.dump(scaler,f)

## ANN Implementation

In [128]:
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [201]:
model = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(64, activation='relu6'),
        Dense(32, activation='relu6'),
        Dense(1, activation='sigmoid')
    ]
)

model.summary()

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_89 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_90 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_91 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [202]:
## Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [203]:
## Setting up TensorBoard directory
LOG_DIR = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

## Setting up TensorBoard
callback = TensorBoard(log_dir=LOG_DIR, histogram_freq=1)

## Setting up EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [205]:
## Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[callback, early_stopping]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8679 - loss: 0.3147 - val_accuracy: 0.8660 - val_loss: 0.3340
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8709 - loss: 0.3132 - val_accuracy: 0.8665 - val_loss: 0.3338
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8689 - loss: 0.3138 - val_accuracy: 0.8590 - val_loss: 0.3357
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8701 - loss: 0.3135 - val_accuracy: 0.8640 - val_loss: 0.3341
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8709 - loss: 0.3107 - val_accuracy: 0.8645 - val_loss: 0.3375
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8717 - loss: 0.3113 - val_accuracy: 0.8605 - val_loss: 0.3504
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8729 - loss: 0.3104 - val_accuracy: 0.8575 - val_loss: 0.3443
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8733 - loss: 0.3087 - val_accu

In [206]:
## Save Model
model.save('churn_model.h5')

In [208]:
## Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 71941), started 0:00:54 ago. (Use '!kill 71941' to kill it.)